In [1]:
from pathlib import Path

ROOT = Path.cwd().parent
DATA = ROOT / "data"
RANDOM_STATE = 42

In [2]:
import pandas as pd
from sklearn.metrics import f1_score


def compute_f1_scores(y_true: pd.DataFrame, y_pred: pd.DataFrame) -> pd.Series:
    """Return the F1 of each label column and the macro F1 (their mean)."""
    scores = pd.Series(f1_score(y_true, y_pred, average=None), index=y_true.columns)
    scores["macro_f1"] = scores.mean()
    return scores

In [3]:
train = pd.read_csv(DATA / "train.csv")
label_columns = [column for column in train.columns if column != "image_id"]
y_true = train[label_columns]

named_pet_columns = ["Moris", "Motya", "Biatrix"]
predictions = {
    "all ones": pd.DataFrame(1, index=y_true.index, columns=label_columns),
    "perfect": y_true.copy(),
    "perfect Cat/Dog, zeros for named pets": y_true.assign(**{column: 0 for column in named_pet_columns}),
}

pd.DataFrame({name: compute_f1_scores(y_true, y_pred) for name, y_pred in predictions.items()}).T

,Cat,Dog,Moris,Motya,Biatrix,macro_f1
all ones,0.585684,0.581429,0.009094,0.006828,0.006666,0.23794
perfect,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000
"perfect Cat/Dog, zeros for named pets",1.000000,1.000000,0.000000,0.000000,0.000000,0.40000


In [4]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

THRESHOLDS = np.arange(0.05, 0.96, 0.05).round(2)

embeddings = np.load(DATA / "embeddings" / "train.npy")
folds = pd.read_csv(ROOT / "folds.csv")["fold"].to_numpy()


def predict_label_probabilities_out_of_fold(embeddings: np.ndarray, label: np.ndarray, folds: np.ndarray) -> np.ndarray:
    """Return P(label=1) for each image, from a model that did not train on the image's fold."""
    probabilities = np.zeros(len(label))
    for fold in np.unique(folds):
        is_valid = folds == fold
        model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
        model.fit(embeddings[~is_valid], label[~is_valid])
        probabilities[is_valid] = model.predict_proba(embeddings[is_valid])[:, 1]
    return probabilities


def predict_probabilities_out_of_fold(embeddings: np.ndarray, y_true: pd.DataFrame, folds: np.ndarray) -> pd.DataFrame:
    """Return the out-of-fold probability of every label column."""
    return pd.DataFrame({
        column: predict_label_probabilities_out_of_fold(embeddings, y_true[column].to_numpy(), folds)
        for column in y_true.columns
    })


def score_thresholds(y_true: pd.DataFrame, probabilities: pd.DataFrame, thresholds: np.ndarray) -> pd.DataFrame:
    """Return one row per threshold with the F1 of each label and the macro F1."""
    return pd.DataFrame({
        threshold: compute_f1_scores(y_true, (probabilities > threshold).astype(int)) for threshold in thresholds
    }).T

In [5]:
logreg_probabilities = predict_probabilities_out_of_fold(embeddings, y_true, folds)

In [6]:
score_thresholds(y_true, logreg_probabilities, THRESHOLDS)

,Cat,Dog,Moris,Motya,Biatrix,macro_f1
0.05,0.997441,0.994353,0.901639,0.898876,0.921348,0.942732
0.10,0.997144,0.994746,0.916667,0.930233,0.921348,0.952028
0.15,0.997144,0.995534,0.904348,0.915663,0.921348,0.946807
0.20,0.997044,0.995830,0.909091,0.902439,0.953488,0.951579
0.25,0.997241,0.995928,0.917431,0.913580,0.976190,0.960074
0.30,0.997437,0.996719,0.915888,0.900000,0.987952,0.959599
0.35,0.997436,0.996719,0.895238,0.886076,1.000000,0.955094
0.40,0.997436,0.996818,0.891089,0.883117,1.000000,0.953692
0.45,0.997338,0.996917,0.891089,0.868421,1.000000,0.950753
0.50,0.997338,0.997016,0.868687,0.853333,1.000000,0.943275


In [7]:
def predict_with_threshold_chosen_on_other_folds(
    y_true: pd.DataFrame, probabilities: pd.DataFrame, folds: np.ndarray, thresholds: np.ndarray
) -> tuple[pd.DataFrame, pd.Series]:
    """Return 0/1 predictions and the chosen threshold per fold; each fold uses the best macro F1 threshold of the other folds."""
    predictions = pd.DataFrame(0, index=y_true.index, columns=y_true.columns)
    chosen_thresholds = {}
    for fold in np.unique(folds):
        is_held_out = folds == fold
        other_scores = score_thresholds(y_true[~is_held_out], probabilities[~is_held_out], thresholds)
        chosen_thresholds[fold] = other_scores["macro_f1"].idxmax()
        predictions.loc[is_held_out] = (probabilities[is_held_out] > chosen_thresholds[fold]).astype(int).to_numpy()
    return predictions, pd.Series(chosen_thresholds, name="threshold")

In [8]:
FIXED_THRESHOLD = 0.5

nested_predictions, chosen_thresholds = predict_with_threshold_chosen_on_other_folds(
    y_true, logreg_probabilities, folds, THRESHOLDS
)
fixed_predictions = (logreg_probabilities > FIXED_THRESHOLD).astype(int)
pd.DataFrame({
    "chosen on other folds": compute_f1_scores(y_true, nested_predictions),
    f"fixed {FIXED_THRESHOLD}": compute_f1_scores(y_true, fixed_predictions),
}).T

,Cat,Dog,Moris,Motya,Biatrix,macro_f1
chosen on other folds,0.997339,0.996027,0.917431,0.900000,0.97619,0.957398
fixed 0.5,0.997338,0.997016,0.868687,0.853333,1.00000,0.943275


In [ ]:
missed_photos = ((y_true == 1) & (fixed_predictions == 0)).sum()
pd.DataFrame({"photos with label": y_true.sum(), f"missed at {FIXED_THRESHOLD}": missed_photos}).T

In [9]:
chosen_thresholds

0    0.25
1    0.25
2    0.25
3    0.25
4    0.30
Name: threshold, dtype: float64

In [10]:
def compute_one_photo_effect(y_true: pd.DataFrame, y_pred: pd.DataFrame) -> pd.DataFrame:
    """Return, per label, the F1 change if one more photo is missed or one more false alarm is raised, and the macro F1 change."""
    true_positive = ((y_true == 1) & (y_pred == 1)).sum()
    false_positive = ((y_true == 0) & (y_pred == 1)).sum()
    false_negative = ((y_true == 1) & (y_pred == 0)).sum()

    def f1(tp, fp, fn):
        return 2 * tp / (2 * tp + fp + fn)

    base_f1 = f1(true_positive, false_positive, false_negative)
    one_more_miss = f1(true_positive - 1, false_positive, false_negative + 1) - base_f1
    one_more_false_alarm = f1(true_positive, false_positive + 1, false_negative) - base_f1
    return pd.DataFrame({
        "f1": base_f1,
        "one more miss": one_more_miss,
        "one more false alarm": one_more_false_alarm,
        "one more miss, macro F1": one_more_miss / len(y_true.columns),
    }).T

In [11]:
FINAL_THRESHOLD = 0.25

final_predictions = (logreg_probabilities > FINAL_THRESHOLD).astype(int)
compute_one_photo_effect(y_true, final_predictions)

,Cat,Dog,Moris,Motya,Biatrix
f1,0.997241,0.995928,0.917431,0.913580,0.976190
one more miss,-0.000099,-0.000100,-0.010024,-0.013580,-0.012335
one more false alarm,-0.000098,-0.000099,-0.008340,-0.011141,-0.011485
"one more miss, macro F1",-0.000020,-0.000020,-0.002005,-0.002716,-0.002467


In [14]:
PARENT_LABELS = {"Moris": "Cat", "Motya": "Cat", "Biatrix": "Dog"}


def set_parent_label_for_named_pets(y_pred: pd.DataFrame) -> pd.DataFrame:
    """Return predictions where a predicted named pet forces its Cat or Dog label to 1."""
    fixed = y_pred.copy()
    for pet, parent in PARENT_LABELS.items():
        fixed.loc[y_pred[pet] == 1, parent] = 1
    return fixed

In [15]:
MACRO_F1_LIMIT = 0.005

parent_predictions = set_parent_label_for_named_pets(final_predictions)
rule_scores = pd.DataFrame({
    "as predicted": compute_f1_scores(y_true, final_predictions),
    "named pet sets Cat/Dog": compute_f1_scores(y_true, parent_predictions),
}).T
rule_scores.loc["gain"] = rule_scores.loc["named pet sets Cat/Dog"] - rule_scores.loc["as predicted"]
print(f"Gain is above the limit {MACRO_F1_LIMIT}:", rule_scores.loc["gain", "macro_f1"] > MACRO_F1_LIMIT)
rule_scores

Gain is above the limit 0.005: False


,Cat,Dog,Moris,Motya,Biatrix,macro_f1
as predicted,9.972408e-01,0.995928,0.917431,0.91358,0.97619,0.960074
named pet sets Cat/Dog,9.972414e-01,0.996028,0.917431,0.91358,0.97619,0.960094
gain,5.436777e-07,0.000100,0.000000,0.00000,0.00000,0.000020


In [16]:
is_changed = parent_predictions != final_predictions
pd.DataFrame({
    "photos changed": is_changed.sum(),
    "now correct": (is_changed & (y_true == 1)).sum(),
    "now wrong": (is_changed & (y_true == 0)).sum(),
}).T

,Cat,Dog,Moris,Motya,Biatrix
photos changed,2,1,0,0,0
now correct,1,1,0,0,0
now wrong,1,0,0,0,0


In [17]:
from sklearn.metrics import average_precision_score, roc_auc_score


def compute_threshold_free_scores(y_true: pd.DataFrame, probabilities: pd.DataFrame) -> pd.DataFrame:
    """Return one row per metric (PR AUC, ROC AUC), with a column per label and the macro mean."""
    scores = pd.DataFrame({
        "PR AUC": {column: average_precision_score(y_true[column], probabilities[column]) for column in y_true.columns},
        "ROC AUC": {column: roc_auc_score(y_true[column], probabilities[column]) for column in y_true.columns},
    }).T
    scores["macro"] = scores.mean(axis=1)
    return scores

In [18]:
f1_row = compute_f1_scores(y_true, final_predictions).rename({"macro_f1": "macro"})
pd.concat([f1_row.to_frame(f"F1 at {FINAL_THRESHOLD}").T, compute_threshold_free_scores(y_true, logreg_probabilities)])

,Cat,Dog,Moris,Motya,Biatrix,macro
F1 at 0.25,0.997241,0.995928,0.917431,0.913580,0.97619,0.960074
PR AUC,0.999899,0.999694,0.980506,0.976632,1.00000,0.991346
ROC AUC,0.999928,0.999694,0.999892,0.999910,1.00000,0.999885


## Conclusions

- The two perfect cases give trivial scores. The first copies the true labels and scores 1.0 on every label. The second scores 1.0 on Cat and Dog and 0.0 on the 3 named pets, so macro F1 is 0.4 (2 of 5 labels). Both match a hand calculation, so the function is correct.
- The all-ones case is a silly baseline. It scores about 0.58 on Cat and Dog and near 0 on the named pets, so macro F1 is only about 0.24.
- Plain accuracy or a micro average would hide this weakness, because Cat and Dog dominate the data. Macro F1 gives each label 20%, so the 3 rare named pets cannot be ignored. We need macro F1.

**Logistic regression baseline**
- We trained one logistic regression for each label on the embeddings. Each image is predicted by a model that did not train on its fold, so the score is out-of-fold.
- With the threshold 0.5, the macro F1 is 0.943.
- Almost all of the macro F1 loss (1 - 0.943) comes from the named pets. Cat and Dog score 0.997, and Biatrix scores 1.0. Moris scores 0.869, and Motya scores 0.853.
- At 0.5, Moris misses 13 of 56 photos, and Motya misses 10 of 42.
- We tried all thresholds from 0.05 to 0.95 with a step of 0.05. The best is 0.25, with macro F1 0.96. The result is not fair, because we choose the threshold on the same data that we score.

**Fair check of the threshold**
- For each fold, we chose the best threshold on the other 4 folds. Then we applied it to the held-out fold only.
- The macro F1 of these predictions is 0.957. The gain over the fixed 0.5 is 0.014.
- The threshold 0.25 was chosen in 4 folds, and 0.3 in the fifth one. So 0.25 is stable.
- The gain comes from Moris and Motya. Biatrix gets slightly worse.
- The folds reuse the stored out-of-fold probabilities. The held-out labels reach the threshold choice in an indirect way. The effect is probably small.

**Effect of one photo**
- At the threshold 0.25, one more missed photo lowers the F1 of Cat or Dog by about 0.0001. It lowers the F1 of a named pet by 0.010 to 0.014.
- For macro F1, one missed photo of a named pet costs 0.0020 (Moris), 0.0027 (Motya) and 0.0025 (Biatrix). The mean is 0.0024.
- We call two macro F1 results similar if they differ by less than 0.005. This is about 2 photos of a named pet.
- A gain smaller than 0.005 is not a real gain. It can come from 2 photos.

**Drawback of macro F1**
- We train 5 separate models, one for each label. So they can give a combination that is impossible, for example Biatrix = 1 and Dog = 0.
- Macro F1 scores each label column alone. It does not see the rows, so it does not see these impossible combinations.

**Rule: a named pet sets Cat or Dog**
- Moris and Motya are always cats, and Biatrix is always a dog. So we tried a rule: if a named pet is predicted 1, set Cat or Dog to 1.
- The rule changes 3 photos of 12260. Two become correct (1 for Cat, 1 for Dog). One becomes wrong (Cat).
- The net gain is 1 correct photo. The macro F1 gain is 0.00002, far below the limit 0.005.
- The rule does not bring a real gain, and it makes the model more complex. We keep the simpler solution and do not use it.
- The reverse rule is not tested: if Cat or Dog is 0, set the named pet to 0.
- It acts on the same 3 photos, so the result is the opposite: 1 photo gained and 2 photos lost.
- The 2 lost photos belong to a named pet, and one photo of a named pet costs about 100 times more than one photo of Cat or Dog.
- So the reverse rule is likely worse than the direct rule. This is a forecast, not a measurement.

**Metrics without a threshold: PR AUC and ROC AUC**
- All scores are very high. The embedding model is large and pretrained on ImageNet, so our task is easy for it.
- ROC AUC is 0.999 or higher for every label. Moris scores higher than Dog, but Moris is much harder by F1.
- The cause is the false positive rate. It divides the false alarms by all negatives, and each named pet has about 12000 negatives. A few false alarms cost almost nothing.
- So ROC AUC shows no difference between the labels. We do not report it.
- PR AUC uses precision. Precision counts only the photos above the threshold, so the easy negatives do not change it.
- PR AUC is lower for Moris (0.981) and Motya (0.977) than for Cat and Dog. Some other photos get a higher probability than some true Moris or Motya photos.
- PR AUC is the only one of the two that shows which labels are hard.

**Why PR AUC next to F1**
- PR AUC compares models without a threshold. The threshold alone moved macro F1 by 0.014 with the same model. With F1 only, a better threshold can look like a better model.
- PR AUC finds the cause of a loss. Biatrix has PR AUC 1.0, but F1 0.976. The model ranks the photos perfectly, so only the threshold causes the loss. A better threshold can fix it.
- Moris has PR AUC 0.981. Some wrong photos rank above some true Moris photos. No threshold can fix all errors. Only a better model can.

**Edge case of F1**
- F1 is 0 divided by 0 when a label has no true photos and no predicted photos. In our setup, sklearn returns 0 with a warning.
- Then a perfect answer loses 0.2 of macro F1, because one label of 5 scores 0.
- In our folds this cannot happen. Each fold has 8 to 12 photos of each named pet.
- In a batch of 32 photos it happens often. So we compute F1 only on the full validation set.

**Decision**
- We use the threshold 0.25.
- We propose the macro PR AUC as the alternative target metric. It keeps 20% for each label, so the rare classes count. It needs no threshold, and the easy negatives do not change it.
- We report the macro PR AUC next to the macro F1.
